# SpaceJEPA v3 · 3. Analysis and report

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rk0604/ReEntryAI/blob/jepa-v3/StochasticEntrySim/ReEntryAI_JEPA_v3_3_analyze.ipynb)

**Runtime: CPU with High-RAM** (it opens that way; nothing here uses a GPU, so a GPU only costs credits).
It takes roughly 30–60 min and releases the runtime by itself when it finishes.

Reads the flights and checkpoints from Drive and writes:
* `03_open_loop/`: footprint error, along-track bias, and ensemble spread per model and seed, on the
  held-out test flights;
* `05_analysis/tables/T##_*.csv` and `figures/F##_*.png`, with `captions.json` saying what each shows
  and which file its data came from;
* `06_report/SpaceJEPA_v3_report.pdf`.

The statistics follow `v3_plan.md` §8, fixed before any v3 flight:
* paired by condition (same IC and the same MPPI noise stream);
* a hierarchical bootstrap with 10,000 resamples (seeds, then cases within each atmosphere);
* per-seed exact McNemar and sign tests;
* Holm correction over the hypothesis-test criteria;
* criteria C1–C7 and the Bridge check.

It can run at any point. With only some seeds finished, it reports those.

## 0. Setup

On Colab this clones the `jepa-v3` branch, installs `pymsis`, and mounts Google Drive. Every output
goes to **`MyDrive/ReEntryAI_jepa_v3/`** (`ReEntryAI_jepa_v3_smoke/` in smoke mode); the layout is in
`jepa/v3_plan.md` §9 and in the auto-written `00_study/README.md`. The 6,000 flights are read from
`MyDrive/ReEntryAI_jepa_data/wide_6000/`. Every stage is resumable: after a disconnect, run all
cells again and finished work is skipped.

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
BRANCH = "jepa-v3"
MODE = os.environ.get("JEPA_V3_MODE", "full")     # "full": the study | "smoke": every stage, tiny, numbers meaningless
if IN_COLAB:
    REPO = "/content/ReEntryAI"
    if not os.path.exists(REPO):
        !git clone -q -b {BRANCH} https://github.com/rk0604/ReEntryAI.git {REPO}
    else:
        !git -C {REPO} pull -q
    %pip install -q pymsis
    os.chdir(f"{REPO}/StochasticEntrySim")
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive"
    STUDY_ROOT = f"{DRIVE}/ReEntryAI_jepa_v3" + ("" if MODE == "full" else f"_{MODE}")
    DATA_CANDIDATES = [f"{DRIVE}/ReEntryAI_jepa_data/wide_6000/episodes_wide_6000.pkl",
                       f"{DRIVE}/ReEntryAI_jepa_data/wide_6000"]
    LOCAL_DATA = "/content/data"
else:
    STUDY_ROOT = os.environ.get("JEPA_V3_ROOT", f"jepa_runs/v3_{MODE}")
    DATA_CANDIDATES = [os.environ.get("JEPA_WIDE_DATA", ""), "jepa_runs/episodes_wide_6000.pkl", "jepa_runs/wide_6000"]
    LOCAL_DATA = "jepa_runs"

import time
from pathlib import Path
SIM_DIR = Path.cwd()
assert (SIM_DIR / "rl_env.py").exists(), "start Jupyter in StochasticEntrySim/"
sys.path.insert(0, str(SIM_DIR))
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from IPython.display import Image, display
%matplotlib inline
from jepa import testset_v3 as TS, study_v3 as S, fly_v3 as F, analyze_v3 as A

CFG = S.StudyConfig.smoke() if MODE == "smoke" else S.StudyConfig()
L = S.Layout(STUDY_ROOT).make()
FROZEN = S.freeze_config(CFG, L)
CPUS = os.cpu_count() or 2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"MODE={MODE}  study folder={STUDY_ROOT}  device={DEVICE}  cpus={CPUS}  git={FROZEN['git_commit'][:10]}")

# ---- Auto-disconnect: stop paying for the runtime once the work is done or a cell fails ----
AUTO_DISCONNECT = IN_COLAB      # set to False to keep the runtime (e.g. to explore results by hand)

def release_runtime(reason):
    # Flush Drive, then release the Colab runtime so it stops using compute units.
    if not (IN_COLAB and AUTO_DISCONNECT):
        print(f"[auto-disconnect off] {reason}")
        return
    print(f"[auto-disconnect] {reason}. Flushing Drive and releasing the runtime; outputs stay visible.", flush=True)
    try:
        drive.flush_and_unmount()
    except Exception as e:
        print("  Drive flush:", e)
    from google.colab import runtime
    runtime.unassign()

def _release_on_error(result):
    err = result.error_in_exec or result.error_before_exec
    if err is not None and not isinstance(err, KeyboardInterrupt):     # pressing stop is not a failure
        release_runtime(f"a cell failed ({type(err).__name__}: {str(err)[:200]})")

_ip = get_ipython()
for _cb in list(_ip.events.callbacks["post_run_cell"]):
    if getattr(_cb, "__name__", "") == "_release_on_error":
        _ip.events.unregister("post_run_cell", _cb)
_ip.events.register("post_run_cell", _release_on_error)
print("auto-disconnect on: the runtime is released when the last cell finishes or a cell fails" if AUTO_DISCONNECT
      else "auto-disconnect off (not on Colab, or switched off): the runtime stays up until you disconnect it")

## 1. Open loop, per model and seed (cached as CSV)

In [ ]:
from jepa import train_v2 as T2
try:
    DATA_PATH = T2.local_copy(T2.find_wide_dataset(DATA_CANDIDATES, download_dir=LOCAL_DATA), LOCAL_DATA)
    test_eps = A.test_split(CFG, DATA_PATH)
    print(f"{len(test_eps)} held-out test flights")
    for seed in CFG.seeds:
        for kind in ("v1", "v2"):
            t0 = time.time()
            if A.open_loop_seed(CFG, L, test_eps, kind, seed) is not None:
                print(f"  {kind} seed {seed}: {time.time() - t0:.0f}s")
except FileNotFoundError as e:
    print("open loop skipped:", e)
OL = A.open_loop_summary(CFG, L)
for k in ("F30", "F31"):
    if k in OL:
        display(Image(str(OL[k])))

## 2. Closed loop: tables, statistics, criteria, figures

In [ ]:
if not (L.testset / "F01_testset_coverage.png").exists():
    A.testset_figures(L, pd.read_csv(SIM_DIR / CFG.testset_csv), pd.read_csv(SIM_DIR / "configs/v3_testset_footprints.csv"))
t0 = time.time()
R = A.analyze(CFG, L)
print(f"analysis of {R.get('n_flights', 0)} flights, seeds {R.get('seeds')}, in {time.time() - t0:.0f}s")
if R:
    pd.set_option("display.max_colwidth", 120)
    display(R["T05"][["id", "question", "measured", "p_holm", "met"]])

In [ ]:
if R:
    cols = ["controller", "atmosphere", "n_seeds", "success5_%", "success5_%_min", "success5_%_max", "miss_median_km",
            "miss_p90_km", "peak_g_mean", "peak_qdot_MWm2_mean", "heat_load_MJm2_mean", "rcs_fuel_kg_mean"]
    display(R["T01"][cols].round(2))
    for k, p in R["figures"].items():
        display(Image(str(p)))

## 3. Report

In [ ]:
if R:
    pdf = A.build_report(CFG, L, R, OL)
    print("report:", pdf)
S.update_manifest(L)
print(open(L.study / "README.md").read())

## Done: release the runtime

Everything this notebook made is already on Drive. This cell flushes Drive and releases the Colab
runtime, so it stops using compute units (a failed cell does the same). To keep the runtime, set
`AUTO_DISCONNECT = False` in the setup cell before running.

In [ ]:
release_runtime("all cells finished")